In [1]:
!pip install --upgrade pip
# Force specific versions to resolve the conflict between TensorFlow and COMET
!pip install "protobuf<5.0.0,>=4.24.4" "tensorflow<2.16" unbabel-comet

# Verification of versions
import google.protobuf
import tensorflow as tf
print(f"Protobuf version: {google.protobuf.__version__}")
print(f"TensorFlow version: {tf.__version__}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 59.1 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
ERROR: Could not find a version that satisfies the requirement tensorflow<2.16 (from versions: 2.16.0rc0, 2.16.1, 2.16.2, 2.17.0rc0, 2.17.0rc1, 2.17.0, 2.17.1, 2.18.0rc0, 2.18.0rc1, 2.18.0rc2, 2.18.0, 2.18.1, 2.19.0rc0, 2.19.0, 2.19.1, 2.20.0rc0, 2.20.0, 2.21.0rc0, 2.21.0rc1, 2.21.0)
ERROR: No matching distribution found for tensorflow<2.16
Protobuf version: 5.29.6
TensorFlow version: 2.20.0


In [3]:
!pip install -q unbabel-comet

import os
# Use the pure-python implementation of protobuf to bypass C++ version mismatch errors
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"

try:
    from comet import download_model, load_from_checkpoint
    print("COMET imported successfully")
except Exception as e:
    print(f"Import failed: {e}")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
tifffile 2026.4.11 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
grain 0.2.16 requires protobuf>=5.28.3, but you have protobuf 4.25.9 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 4.25.9 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 4.25.9 

COMET imported successfully


In [4]:
# =========================================================
# COMET / AfriCOMET Evaluation
# Fine-Tuned AfriNLLB Model
# =========================================================
# !pip uninstall -y protobuf -q
# !pip uninstall -y google -q
# !pip install -q --no-cache-dir p protobuf==3.20.3 "numpy==1.26.4"
# # Install COMET if not already installed
# !pip install -q --no-cache-dir "unbabel-comet==2.2.2" sacrebleu sentencepiece

import json
import os

# =========================================================
# Save predictions from fine-tuned model
# =========================================================

with open("retrained_predictions.json", "r", encoding="utf-8") as f:
    predictions = json.load(f)

with open("retrained_sources.json", "r", encoding="utf-8") as f:
    sources = json.load(f)

with open("retrained_references.json", "r", encoding="utf-8") as f:
    references = json.load(f)

print("Loaded:")
print("Predictions:", len(predictions))
print("Sources:", len(sources))
print("References:", len(references))

Loaded:
Predictions: 1012
Sources: 1012
References: 1012


In [5]:
from google.colab import files
files.upload() # upload kaggle.json
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d mathurinache/flores101
!unzip -q flores101.zip -d flores101
!ls flores101

# import kagglehub

# # Download latest version
# base_path = kagglehub.dataset_download("mathurinache/flores101")

# print("FLORES dataset ready.")
# print("Base path:", base_path)

Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/mathurinache/flores101
License(s): CC-BY-NC-SA-4.0
100% 13.0M/13.0M [00:02<00:00, 6.81MB/s]

flores101_dataset


In [6]:
# =========================================================
# Reload predictions + FLORES evaluation data
# =========================================================

import json

# Load baseline predictions
with open("retrained_predictions.json", encoding="utf-8") as f:
    baseline_predictions = json.load(f)

# Load RAT predictions
with open("rat_predictions.json", encoding="utf-8") as f:
    rat_predictions = json.load(f)

# Load Constrained predictions
with open("constrained_predictions.json", encoding="utf-8") as f:
    cons_predictions = json.load(f)

# Load post processing predictions
with open("post_processing_predictions.json", encoding="utf-8") as f:
    post_processing_predictions = json.load(f)

# Reload FLORES data
def load_lines(path):
    with open(path, "r", encoding="utf-8") as f:
        return [line.strip() for line in f]

base_path = "/content/flores101/flores101_dataset/devtest"

sources = load_lines(f"{base_path}/eng.devtest")
references = load_lines(f"{base_path}/zul.devtest")

print("Loaded evaluation data:")
print("Sources:", len(sources))
print("References:", len(references))
print("Baseline Predictions:", len(baseline_predictions))
print("RAT Predictions:", len(rat_predictions))

Loaded evaluation data:
Sources: 1012
References: 1012
Baseline Predictions: 1012
RAT Predictions: 1012


In [7]:
# =========================================================
# Load AfriCOMET Model
# =========================================================

from comet import download_model, load_from_checkpoint

# Try primary AfriCOMET checkpoint
try:
    model_path = download_model("masakhane/africomet")
    print("Loaded primary AfriCOMET checkpoint")

except:
    print("Primary checkpoint unavailable.")
    print("Trying fallback checkpoint...")

    model_path = download_model("masakhane/africomet-qe-stl")

# Load model
comet_model = load_from_checkpoint(model_path)

print("AfriCOMET model ready.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Primary checkpoint unavailable.
Trying fallback checkpoint...


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

checkpoints/model.ckpt:   0%|          | 0.00/2.30G [00:00<?, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/714 [00:00<?, ?B/s]

AfriCOMET model ready.


In [8]:
baseline_data = [
    {
        "src": s,
        "mt": p,
        "ref": r
    }
    for s, p, r in zip(sources, baseline_predictions, references)
]

rat_data = [
    {
        "src": s,
        "mt": p,
        "ref": r
    }
    for s, p, r in zip(sources, rat_predictions, references)
]

const_data = [
    {
        "src": s,
        "mt": p,
        "ref": r
    }
    for s, p, r in zip(sources, cons_predictions, references)
]


post_data = [
    {
        "src": s,
        "mt": p,
        "ref": r
    }
    for s, p, r in zip(sources, post_processing_predictions, references)
]


print("Prepared evaluation samples:")
print("Baseline:", len(baseline_data))
print("RAT:", len(rat_data))

Prepared evaluation samples:
Baseline: 1012
RAT: 1012


In [9]:
# =========================================================
# Run AfriCOMET evaluation
# =========================================================

baseline_result = comet_model.predict(
    baseline_data,
    batch_size=8,
    gpus=1
)

rat_result = comet_model.predict(
    rat_data,
    batch_size=8,
    gpus=1
)

const_result = comet_model.predict(
    const_data,
    batch_size=8,
    gpus=1
)

post_result = comet_model.predict(
    post_data,
    batch_size=8,
    gpus=1
)

baseline_score = baseline_result["system_score"]
rat_score = rat_result["system_score"]
const_score = const_result["system_score"]
post_score = post_result["system_score"]

print("Baseline AfriCOMET:", baseline_score)
print("RAT Refined AfriCOMET:", rat_score)
print("Constrained Refined AfriCOMET:", const_score)
print("Post processing refined AfriCOMET:", post_score)

INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
Predicting DataLoader 0: 100%|██████████| 127/127 [19:24<00:00,  9.17s/it]
INFO:pytorch_lightning.utilities.rank_zero:

Baseline AfriCOMET: 0.7171128208458188
RAT Refined AfriCOMET: 0.7261376729006824
Constrained Refined AfriCOMET: 0.493612851947546
Post processing refined AfriCOMET: 0.7151843064092835


In [15]:
# =========================================================
# Save final metrics
# =========================================================

metrics = {
    "fineTuned_africomet": baseline_score,
    "rat_africomet": rat_score,
    "improvement": rat_score - baseline_score,
    "constrained_africomet": const_score,
    "post_africomet": post_score,
}

with open("comparative_africomet_results.json", "w") as f:
    json.dump(metrics, f, indent=2)

print("Saved AfriCOMET")

Saved AfriCOMET
